# Task 1.3 — Baseline Comparison

**Paper:** *Analysis of SVM with Indefinite Kernels* (NIPS 2009)  
**Authors:** Yiming Ying, Colin Campbell, Mark Girolami

---

This notebook compares the proposed method against the baseline, identifies limitations of the baseline, explains how the paper improves upon it, and describes scenarios where the proposed method may fail.

---

## Baseline Method: Standard SVM with PSD Kernel

The baseline approach is the **standard Support Vector Machine** with a valid positive semi-definite (PSD) kernel function, such as:

- **RBF (Gaussian) kernel:** $K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$
- **Linear kernel:** $K(x_i, x_j) = x_i^T x_j$
- **Polynomial kernel:** $K(x_i, x_j) = (x_i^T x_j + c)^d$

These kernels satisfy Mercer's condition (PSD), which guarantees:
1. The dual problem is **convex** (quadratic programming)
2. **Strong duality** holds (KKT conditions are necessary and sufficient)
3. A **unique global optimum** exists
4. Efficient solvers (e.g., SMO, LIBSVM) are applicable

The standard SVM dual:

$$\max_{\alpha} \sum_{i=1}^n \alpha_i - \frac{1}{2} \alpha^T Y K Y \alpha$$

subject to $0 \leq \alpha_i \leq C$, $\sum_i \alpha_i y_i = 0$, where $Y = \text{diag}(y_1, \ldots, y_n)$.

---

## Limitations of the Baseline

### 1. Cannot Handle Indefinite Kernels
The most critical limitation: standard SVMs **require** PSD kernels. Many real-world similarity measures are indefinite:
- **Sigmoidal kernel:** $K(x_i, x_j) = \tanh(\kappa x_i^T x_j + c)$ — not PSD for all parameters
- **Hausdorff distance kernels** — used in shape matching, often indefinite
- **Power spectrum kernels** — used in bioinformatics, can be indefinite
- **Edit-distance based similarity** — commonly indefinite

### 2. Non-Convexity Issues
If an indefinite kernel is naively used in the standard SVM dual:
- The objective is **non-convex** (indefinite quadratic form)
- QP solvers may converge to **local optima** or fail entirely
- No theoretical guarantees on solution quality

### 3. Ad-hoc Fixes Are Unsatisfying
Common workarounds for indefinite kernels in standard SVMs include:
- **Spectrum clipping:** Manually set negative eigenvalues to zero — lacks theoretical justification
- **Spectrum shifting:** Add $\lambda I$ to make $K + \lambda I \succeq 0$ — changes the kernel significantly
- **Ignoring the issue:** Use the indefinite kernel and hope for the best — unreliable

None of these have formal optimality or convergence guarantees.

---

## How the Paper Improves Upon the Baseline

### 1. Principled Min-Max Framework
Instead of ad-hoc fixes, the paper provides a **theoretically grounded** formulation:

$$\min_{\alpha \in \mathcal{A}} \max_{\tilde{K} \in \mathcal{K}} f(\alpha, \tilde{K})$$

This jointly optimizes the SVM classifier and PSD kernel proxy.

### 2. Convergence Guarantees
- **Projected Gradient:** $O(1/t)$ convergence rate
- **Nesterov Acceleration:** $O(1/t^2)$ convergence rate (optimal for first-order methods)
- These are **provable guarantees**, unlike heuristic approaches

### 3. Automatic PSD Proxy Learning
The optimal PSD proxy $K_{+}$ is determined automatically by the optimization, rather than requiring manual tuning or domain expertise.

### 4. Theoretically Justified
The paper provides:
- **Generalization bounds** for the learned classifier
- **Connection to RKHS theory** even for indefinite kernels
- **Consistency results** showing the method converges to the optimal solution

### Comparison Table

| Aspect | Standard SVM | Proposed Method |
|--------|-------------|----------------|
| Kernel requirement | PSD only | Any (indefinite OK) |
| Optimization | Convex QP | Convex (after projection) |
| Convergence guarantee | Yes (convex QP) | Yes ($O(1/t^2)$) |
| Handles indefinite kernels | No | Yes |
| Computational cost | $O(n^2)$–$O(n^3)$ | $O(n^3)$ (eigen-decomp) |
| Theoretical justification | Mercer's theorem | Min-max framework |

---

## Scenario Where the Proposed Method Fails

### Failure Case: Very Large Datasets

**Scenario:**  
Consider a dataset with $n = 100{,}000$ samples. The kernel matrix $K \in \mathbb{R}^{100000 \times 100000}$ requires:
- **Memory:** $100{,}000^2 \times 8$ bytes $\approx$ **74.5 GB** (float64)
- **Eigen-decomposition:** $O(n^3) = O(10^{15})$ operations — **infeasible** on standard hardware

The method's reliance on full eigen-decomposition of the kernel matrix makes it impractical for large-scale problems.

### Additional Failure Cases

**1. Heavily Noisy Kernels:**  
When the kernel matrix is dominated by noise (very low signal-to-noise ratio), the PSD projection may remove most of the structure, leaving an uninformative kernel.

**2. Negative Eigenvalues Carry Information:**  
In some domains (e.g., non-metric spaces), the negative eigenspace may encode crucial discriminative information that is lost during PSD projection.

**3. Choice of $\rho$ is Critical:**  
The radius $\rho$ of the constraint set $\mathcal{K}$ controls how far the PSD proxy can deviate from the original kernel. Poor choice of $\rho$ leads to either:
- **Too small $\rho$:** Proxy is forced to be very close to $K$ but still PSD → may not be achievable or leads to a trivial proxy
- **Too large $\rho$:** Proxy can be any PSD matrix → loses connection to the original kernel

---

## Summary

The paper significantly advances the state of the art for kernel methods with indefinite kernels by providing a principled, theoretically justified framework with convergence guarantees. However, its computational scalability remains a limitation, and its effectiveness depends on the assumption that PSD projection preserves the discriminative structure of the kernel. Future work could address these limitations through approximate eigen-decomposition methods (e.g., Nyström approximation) or alternative kernel correction strategies.